# Artificial Neural Network using Keras and TensorFlow

**Student:** Shubh Routh  
**Topic:** Artificial Neural Networks (ANN)  
**Course:** Machine Learning

## Introduction

In this activity, I am using Keras with TensorFlow to build a small neural-network classifier for tabular data.

The aim is to understand the practical steps behind an ANN rather than treating the model as a black box. I will prepare the data, create separate training, validation and test sets, scale the numerical features, build a dense neural network, train it, check its learning curves and evaluate its final predictions.

The notebook also includes an improved version using dropout so that the effect of a simple regularization technique can be discussed.


## Learning Objectives

After completing this activity, I should be able to:

- explain why training, validation and test data are kept separate;
- understand why feature scaling is useful for neural-network training;
- identify the role of dense layers and activation functions;
- explain the purpose of the loss function and optimizer;
- understand epochs, batch size and early stopping;
- evaluate an ANN using accuracy, ROC-AUC, a confusion matrix and a classification report; and
- explain how dropout can help control overfitting.


## 1. Importing the Required Libraries

The first cell loads the libraries needed for this experiment.

TensorFlow and Keras are used to create and train the neural networks. Pandas and NumPy support data handling, while Matplotlib and Seaborn are used for the result visualizations. Scikit-learn provides the dataset, preprocessing utilities and evaluation metrics.

Keeping the imports together makes the notebook easier to follow when revisiting it later.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("TensorFlow version:", tf.__version__)


## 2. Loading the Dataset

For this exercise, I am using scikit-learn's built-in **Breast Cancer Wisconsin Diagnostic dataset**.

It is a binary classification problem. The features describe characteristics calculated from digitized images of breast-mass samples, and the target represents the two diagnostic classes.

Using a built-in dataset keeps the focus on the ANN itself and avoids spending most of the activity on downloading and cleaning an external file.


In [ ]:
dataset = load_breast_cancer()

features = pd.DataFrame(
    dataset.data,
    columns=dataset.feature_names
)
target = pd.Series(
    dataset.target,
    name="target"
)

print("Dataset shape:", features.shape)
print("Number of classes:", target.nunique())

display(features.head())


### Observation

The dataset contains **569 observations and 30 numerical features**, with two target classes. The features are already numerical, so the main preprocessing requirement for this ANN experiment is putting them on comparable scales.


## 3. Creating Training, Validation and Test Sets

Instead of using one split for everything, three groups are used:

- **Training data** is used to learn the network parameters.
- **Validation data** is used while training to monitor generalization.
- **Test data** is kept aside until the end for the final evaluation.

This separation gives a fairer picture of how the trained network performs on observations it did not use for fitting.


In [ ]:
X_train, X_remaining, y_train, y_remaining = train_test_split(
    features,
    target,
    test_size=0.30,
    random_state=42,
    stratify=target
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42,
    stratify=y_remaining
)

print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)
print("Test set:", X_test.shape)


### Observation

The 70:15:15 arrangement gives the ANN enough observations for learning while still keeping separate validation and test data. Stratification is used so that both classes remain represented in each split.


## 4. Standardizing the Features

The features in a tabular dataset can have very different numerical ranges. A neural network can train more smoothly when the input variables are standardized.

The scaler is fitted **only on the training data**. The same transformation is then applied to the validation and test sets. This avoids allowing information from the evaluation data to influence the preprocessing step.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)


### Observation

After scaling, the three datasets retain their original number of observations and features, but their numerical values are now on a more comparable scale. This gives the ANN a more suitable input for optimization.


## 5. Building the First ANN

The first network is a simple feedforward model made from dense layers.

The structure is:

**Input → 32 ReLU neurons → 16 ReLU neurons → 1 sigmoid output**

ReLU is used in the hidden layers to introduce nonlinearity. Since the target has two classes, the final sigmoid neuron produces a value that can be interpreted as the probability of one class.


In [ ]:
tf.random.set_seed(42)

baseline_ann = keras.Sequential([
    keras.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

baseline_ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

baseline_ann.summary()


### Observation

The network has two hidden dense layers followed by a single sigmoid output neuron. The binary cross-entropy loss is appropriate for the binary target, while Adam is used to update the model parameters during training.


## 6. Training the Baseline Model

The model will be trained for a maximum of 100 epochs. The validation data is monitored during training.

Early stopping is included so that training can stop when validation loss no longer improves. Restoring the best weights means the final model uses the parameter values from the strongest validation point rather than simply the last epoch.


In [ ]:
stop_when_stuck = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

baseline_history = baseline_ann.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_valid_scaled, y_valid),
    epochs=100,
    batch_size=32,
    callbacks=[stop_when_stuck],
    verbose=1
)

print("Training epochs completed:", len(baseline_history.history["loss"]))


### Observation

Early stopping allows the network to avoid continuing indefinitely once validation performance stops improving. The number of completed epochs can therefore be lower than the maximum of 100.


## 7. Looking at the Learning Curves

The training history contains the values recorded at each epoch.

I am plotting loss and accuracy separately because the two graphs answer different questions: loss shows how the optimization objective changes, while accuracy gives a more intuitive view of classification performance.


In [ ]:
history = pd.DataFrame(baseline_history.history)

plt.figure(figsize=(8, 5))
plt.plot(history["loss"], label="Training")
plt.plot(history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Baseline ANN: Training and Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["accuracy"], label="Training")
plt.plot(history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline ANN: Training and Validation Accuracy")
plt.legend()
plt.show()


### Observation

The learning curves allow the training and validation behaviour to be compared across epochs. A large and persistent gap between the two curves can be a warning sign of overfitting, while poor performance on both can indicate that the model has not learned enough.


## 8. Evaluating the Baseline ANN

The test set has been kept untouched during model fitting, so it is now used for the final evaluation.

Along with accuracy, ROC-AUC and a classification report are calculated. The predicted probabilities are converted into class labels using a threshold of 0.5.


In [ ]:
test_loss, test_accuracy, test_auc = baseline_ann.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

test_probabilities = baseline_ann.predict(
    X_test_scaled,
    verbose=0
).ravel()

test_predictions = (test_probabilities >= 0.5).astype(int)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=dataset.target_names
    )
)


### Observation

The test metrics summarize how the baseline ANN performs on data that was not used during training. Accuracy gives the overall proportion of correct predictions, while ROC-AUC and the class-level precision, recall and F1-scores provide additional information about the classifier.


## 9. Confusion Matrix and ROC Curve

Two visual checks are used here.

The confusion matrix shows the actual class against the predicted class. The ROC curve shows how the true-positive rate changes relative to the false-positive rate at different probability thresholds.


In [ ]:
matrix = confusion_matrix(
    y_test,
    test_predictions
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=dataset.target_names,
    yticklabels=dataset.target_names
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Baseline ANN Confusion Matrix")
plt.show()

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_test,
    test_probabilities
)

plt.figure(figsize=(7, 5))
plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUC = {roc_auc_score(y_test, test_probabilities):.3f}"
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Baseline ANN ROC Curve")
plt.legend()
plt.show()


### Observation

The confusion matrix makes the types of classification errors easier to see than accuracy alone. The ROC curve provides another view of the model's ability to separate the two classes across different decision thresholds.


## 10. Improving the ANN with Dropout

A second version of the network is created to demonstrate a common regularization method: **dropout**.

During training, dropout temporarily removes a fraction of activations from the network. The intention is to reduce excessive dependence on particular neurons and help control overfitting.

The second model is intentionally kept close enough to the baseline that the effect of the change can be discussed clearly.


In [ ]:
tf.random.set_seed(42)

regularized_ann = keras.Sequential([
    keras.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(1, activation="sigmoid")
])

regularized_ann.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

regularized_history = regularized_ann.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_valid_scaled, y_valid),
    epochs=100,
    batch_size=32,
    callbacks=[stop_when_stuck],
    verbose=0
)

print(
    "Regularized ANN training completed in",
    len(regularized_history.history["loss"]),
    "epochs."
)


### Observation

The second model adds dropout after the hidden layers. This does not guarantee a higher test score; its purpose is to make the model less prone to relying too heavily on particular internal activations and to provide a practical example of regularization.


## 11. Comparing the Two Models

A comparison should be based on the same held-out test data.

The baseline and dropout models are evaluated using the same accuracy and ROC-AUC measures so that the comparison is fair.


In [ ]:
regularized_loss, regularized_accuracy, regularized_auc = regularized_ann.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

comparison = pd.DataFrame({
    "Model": ["Baseline ANN", "ANN with Dropout"],
    "Test Accuracy": [test_accuracy, regularized_accuracy],
    "Test ROC-AUC": [test_auc, regularized_auc]
})

display(comparison.round(4))


### Observation

The comparison table gives a direct view of whether adding dropout helped on this particular test set. The better model should not be chosen from accuracy alone; ROC-AUC and the earlier learning curves should also be considered when interpreting the result.


## 12. Key Takeaways

- Scaling is important because neural networks are trained using numerical optimization.
- Dense layers learn combinations of input features through trainable weights and biases.
- ReLU provides nonlinear transformations in hidden layers.
- A sigmoid output is suitable for a binary classification problem.
- Validation data helps monitor generalization during training.
- Early stopping can prevent unnecessary training once validation performance stops improving.
- Dropout is a regularization technique that can help reduce overfitting.
- Accuracy, precision, recall, F1-score, confusion matrices and ROC-AUC provide complementary views of model performance.


## 13. Conclusion

This activity moved from a basic dense neural network to a regularized version using dropout.

The important part is understanding the complete workflow: prepare the data, scale the features, separate training/validation/test data, define the network, select a loss function and optimizer, train while monitoring validation performance, and finally evaluate the model on unseen test observations.

The experiment also shows why neural-network performance should be interpreted using more than one metric. A single accuracy value does not explain the kinds of mistakes a classifier makes or how its performance changes across probability thresholds.


## Practice Questions

1. Why should the scaler be fitted only on the training data?
2. What is the role of a dense layer in this ANN?
3. Why is ReLU used in the hidden layers?
4. Why is sigmoid used in the final layer for this task?
5. What does binary cross-entropy measure?
6. How does early stopping help during training?
7. What problem is dropout intended to address?
8. Why is ROC-AUC useful in addition to accuracy?
9. What does a confusion matrix tell us that accuracy alone does not?
10. Which model performed better on the held-out test data, and what evidence supports that conclusion?
